In [8]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col, struct, to_json
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType

In [9]:
# Initialize Spark with Kafka package
spark = SparkSession.builder \
 .appName("FraudDetection") \
 .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.3") \
 .getOrCreate()
spark.sparkContext.setLogLevel("WARN")

In [10]:
# 1. Load Static User Data (CSV)
users_df = spark.read.csv(
    "data/user_table.csv",
    header=True,
    inferSchema=True
)

users_df.show()
users_df.printSchema()


+------+-------------+-------------------+--------+--------------+
|userId|         name|              email|   phone|account_status|
+------+-------------+-------------------+--------+--------------+
|   101|  Alice Smith|  alice@example.com|555-0101|        active|
|   102|    Bob Jones|    bob@example.com|555-0102|        active|
|   103|Charlie Brown|charlie@example.com|555-0103|        active|
|   104| Diana Prince|  diana@example.com|555-0104|        active|
|   105|  Evan Wright|   evan@example.com|555-0105|  under_review|
+------+-------------+-------------------+--------+--------------+

root
 |-- userId: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- account_status: string (nullable = true)



In [11]:
# 2. Read Streaming Data from Kafka
tx_schema = StructType([
 StructField("tx_id", IntegerType(), True),
 StructField("userId", IntegerType(), True),
 StructField("amount", DoubleType(), True),
 StructField("timestamp", DoubleType(), True)
])
kafka_stream = spark.readStream \
 .format("kafka") \
 .option("kafka.bootstrap.servers", "kafka:9092") \
 .option("subscribe", "fraud-detection") \
 .load()

In [ ]:
# 3. Parse and Filter Data
parsed_stream = kafka_stream.select(from_json(col("value").cast("string"),
tx_schema).alias("tx")).select("tx.*")
fraud_stream = parsed_stream.filter(col("amount") > 10000.0)
fraud_stram_2 = parsed_stream.filter(col("amount")>5000.0)

In [13]:
# 4. Enrich Stream with User Details
enriched_fraud = fraud_stream.join(users_df, "userId")
enriched_fraud_2 = fraud_stream.join(users_df,"userId")

In [14]:
# 5. Format for output Kafka topic
output_stream = enriched_fraud \
 .withColumn("value", to_json(struct("*")).cast("string")) \
 .select("value")

In [15]:
# 6. Write Stream to 'fraud-notification' Topic
query = output_stream.writeStream \
 .format("kafka") \
 .option("kafka.bootstrap.servers", "kafka:9092") \
 .option("topic", "fraud-notification") \
 .option("checkpointLocation", "/workspace/checkpoints") \
 .start()
query.awaitTermination()

26/06/19 04:34:09 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/06/19 04:34:09 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
26/06/19 04:34:20 WARN NetworkClient: [Producer clientId=producer-1] Error while fetching metadata with correlation id 1 : {fraud-notification=UNKNOWN_TOPIC_OR_PARTITION}
26/06/19 04:34:21 WARN CheckpointFileManager: Failed to rename temp file file:/workspace/checkpoints/offsets/.4.257ce353-0519-4fff-8bc3-9073a8f50db0.tmp to file:/workspace/checkpoints/offsets/4 because file exists
org.apache.hadoop.fs.FileAlreadyExistsException: rename destination file:/workspace/checkpoints/offsets/4 already exists.
	at org.apache.hadoop.fs.FileSystem.rename(FileSystem.java:1600)
	at org.apache.hadoop.fs.DelegateToFileSystem.renameInternal(DelegateToFileSystem.java:

StreamingQueryException: [STREAM_FAILED] Query [id = 8c1cca8e-bd91-4944-b3d4-80eb229a0c30, runId = 51425e9e-9cae-4831-a787-08ef737cf53e] terminated with exception: Multiple streaming queries are concurrently using file:/workspace/checkpoints/offsets.